# YOLOv3 Bottle Detection (still image)

This notebook adapts the original experiment to use local notebook paths. It detects the COCO `bottle` class in a still image; it does not control the robot.

**Before running:** download `yolov3.weights`, `yolov3.cfg`, and `coco.names` from their upstream sources, and upload an image you are allowed to use.


In [ ]:
%pip install opencv-python-headless numpy


## 1. Set paths
Upload the model files and a test image to the Colab session, or change these paths to your mounted Drive locations.

In [ ]:
from pathlib import Path
import cv2
import numpy as np

MODEL_DIR = Path('/content/models/yolov3')
WEIGHTS_PATH = MODEL_DIR / 'yolov3.weights'
CONFIG_PATH = MODEL_DIR / 'yolov3.cfg'
NAMES_PATH = MODEL_DIR / 'coco.names'
IMAGE_PATH = Path('/content/test-image.jpg')

for path in (WEIGHTS_PATH, CONFIG_PATH, NAMES_PATH, IMAGE_PATH):
    print(f'{path}: {path.exists()}')


## 2. Load the model and class names

In [ ]:
if not all(p.is_file() for p in (WEIGHTS_PATH, CONFIG_PATH, NAMES_PATH)):
    raise FileNotFoundError('Upload YOLOv3 weights, config, and coco.names to MODEL_DIR first.')

net = cv2.dnn.readNetFromDarknet(str(CONFIG_PATH), str(WEIGHTS_PATH))
layer_names = net.getLayerNames()
output_layers = [layer_names[int(i) - 1] for i in np.asarray(net.getUnconnectedOutLayers()).flatten()]
classes = [line.strip() for line in NAMES_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'Loaded {len(classes)} classes; bottle class available: {"bottle" in classes}')


## 3. Detect bottles

In [ ]:
def detect_bottles(image, confidence_threshold=0.50, nms_threshold=0.40):
    if image is None or image.size == 0:
        raise ValueError('The image was not loaded correctly.')
    if 'bottle' not in classes:
        raise ValueError("The COCO class list does not contain 'bottle'.")
    height, width = image.shape[:2]
    blob = cv2.dnn.blobFromImage(image, 1/255.0, (416, 416), (0, 0, 0), swapRB=True, crop=False)
    net.setInput(blob)
    outputs = net.forward(output_layers)
    bottle_id = classes.index('bottle')
    boxes, confidences = [], []
    for output in outputs:
        for detection in output:
            confidence = float(detection[4]) * float(detection[5 + bottle_id])
            if confidence < confidence_threshold:
                continue
            cx, cy = int(detection[0] * width), int(detection[1] * height)
            w, h = int(detection[2] * width), int(detection[3] * height)
            x, y = max(0, int(cx - w/2)), max(0, int(cy - h/2))
            w, h = min(w, width-x), min(h, height-y)
            if w > 0 and h > 0:
                boxes.append([x, y, w, h])
                confidences.append(confidence)
    indices = cv2.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold) if boxes else []
    kept = np.asarray(indices).flatten().astype(int).tolist() if len(indices) else []
    return [{'label':'bottle', 'box':boxes[i], 'confidence':confidences[i]} for i in kept]


## 4. Run and visualize

In [ ]:
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f'Could not read image: {IMAGE_PATH}')
detections = detect_bottles(image)
annotated = image.copy()
for item in detections:
    x, y, w, h = item['box']
    label = f"{item['label']}: {item['confidence']*100:.1f}%"
    cv2.rectangle(annotated, (x,y), (x+w,y+h), (0,255,0), 2)
    cv2.putText(annotated, label, (x,max(20,y-8)), cv2.FONT_HERSHEY_SIMPLEX, .55, (0,255,0), 2)
print(f'Detected {len(detections)} bottle(s).')
from google.colab.patches import cv2_imshow
cv2_imshow(annotated)


## Interpretation and limitations

- The model detects the generic COCO `bottle` class; it does not verify plastic material or whether an object is litter.
- This notebook processes a still image only.
- The notebook does not control the mobile base or the robotic arm.
- For a real robot, add a validated safety supervisor, obstacle sensing, depth/pose estimation, and supervised hardware testing.